In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '3'

In [ ]:
import cv2
import json
import glob
import random
import numpy as np
import pandas as pd
from tqdm import tqdm
from pathlib import Path
from copy import deepcopy

import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
import torch
import torchvision
import lightning as L

In [ ]:
from common import plot_one_box, increment_path
from common import random_perspective, load_mosaic, Albumentations
from common import letterbox, xyxy2xywh, xywhn2xyxy, box2segment, scale_boxes
from common_eval import PR_eval, confusion_matrix

# I. Configuration

In [ ]:
# Training parameters
VGG_PATH = os.path.join(os.getcwd(), 'RefineDet/weights/vgg16_reducedfc.pth')
ROOT_DIR = os.path.join(os.getcwd(), 'datasets/PCBdatasets/VOC_format')
CLASSES = [
    "Scratches on the circuit board",
    "Scratches on components",
    "Missing/lost component",
    "Damaged component pins",
    "Faulty components"
]
IMG_SIZE = 512
EPOCHS = 100
BATCH_SIZE = 8

# Evaluation parameters
CONF_THRESH = 0.5
NMS_THRESH = 0.25
IOU_THRESH = 0.5

# II. Load Datasets

In [ ]:
import xml.etree.ElementTree as ET
from torch.utils.data import Dataset, DataLoader

In [ ]:
class RefineDetVOCDataset(Dataset):
    def __init__(self, mode, root_dir, classes, img_size):
        self.mode = mode
        assert self.mode in ['train', 'valid', 'test'], ValueError
        self.albument = Albumentations(p=0.1, box_format='pascal_voc') if self.mode == 'train' else None
        self.to_tensor = torchvision.transforms.ToTensor()
        self.root_dir = root_dir
        self.label2idx = {classes[idx]: idx for idx in range(len(classes))}
        self.idx2label = {idx: classes[idx] for idx in range(len(classes))}
        self.img_size = img_size

        # load data
        self.images_info = self.load_images_and_anns(os.path.join(root_dir, mode), self.label2idx)

    def __len__(self):
        return len(self.images_info)

    def __getitem__(self, index):
        # load data
        if self.mode != 'test':
            if self.mode == 'train':
                # load data and geometric augmentation
                if random.random() < 0.8:
                    im, targets = load_mosaic(
                        load_fn=self.load_data,
                        indices=[index] + random.sample(range(len(self)), k=3),
                        s=self.img_size, degrees=0, translate=0.1, scale=0.2, shear=0,
                        perspective=0.0001, wh_thr=2, ar_thr=100, area_thr=0.01
                    )
                else:
                    im, targets, segments, (h, w) = self.load_data(index)
                    im, ratio, pad = letterbox(im=im, new_shape=2 * [self.img_size], auto=False)
                    segments = [[w, h] * (s * ratio) + pad for s in segments]
                    targets[:, 1:] = xywhn2xyxy(targets[:, 1:], ratio[0] * w, ratio[1] * h, pad[0], pad[1])
                    im, targets, segments = random_perspective(
                        im=im, targets=targets, segments=segments,
                        degrees=0, translate=0.1, scale=0.2, shear=0,
                        perspective=0.0001, wh_thr=2, ar_thr=100, area_thr=0.01
                    )
                if random.random() < 0.5:
                    im = np.ascontiguousarray(im[:, ::-1, :])
                    if len(targets):
                        targets[:, [1, 3]] = self.img_size - targets[:, [3, 1]]
                # color augmentation
                im, targets = self.albument(im, targets)
            elif self.mode == 'valid':
                im, targets, segments, (h, w) = self.load_data(index)
                im, ratio, pad = letterbox(im=im, new_shape=2 * [self.img_size], auto=False)
                segments = [[w, h] * (s * ratio) + pad for s in segments]
                if len(targets):
                    targets[:, 1:] = xywhn2xyxy(targets[:, 1:], ratio[0] * w, ratio[1] * h, pad[0], pad[1])

            if len(targets) == 0:
                targets = np.zeros((0, 5), dtype='float32')

            # format for train
            im_tensor = 255 * self.to_tensor(im)
            targets = {
                'labels': torch.from_numpy(targets[:, 0].astype('int64')),
                'boxes': torch.from_numpy(targets[:, 1:].astype('float32') / self.img_size)
            }
            return im_tensor, targets
        else:
            im_info = self.images_info[index]
            # load image
            im = cv2.cvtColor(src=cv2.imread(im_info['filename']), code=cv2.COLOR_BGR2RGB)
            h, w, _ = im.shape
            # load target
            labels, boxes = [], []
            for det in im_info['detections']:
                labels.append(det['label'])
                boxes.append(det['bbox'])
            targets = {
                'labels': np.array(labels, dtype='int64'),
                'boxes': np.array(boxes, dtype='float32') * [w, h, w, h]
            }
            return im, targets, Path(im_info['filename']).name

    def load_data(self, index):
        im_info = self.images_info[index]
        # load image
        im = cv2.imread(im_info['filename'])
        h0, w0 = im.shape[:2]  # orig hw
        r = self.img_size / max(h0, w0)  # ratio
        if r != 1:
            interp = cv2.INTER_LINEAR if r > 1 else cv2.INTER_AREA
            im = cv2.resize(src=im, dsize=[int(w0 * r), int(h0 * r)], interpolation=interp)
        im = cv2.cvtColor(src=im, code=cv2.COLOR_BGR2RGB)
        targets, segments = [], []
        for det in im_info['detections']:
            targets.append(np.concatenate([[det['label']], det['bbox']], axis=0))
            segments.append(det['bbox_coords'])
        targets = np.array(targets, dtype='float32')
        targets[:, 1:] = xyxy2xywh(targets[:, 1:])
        return im, targets, segments, im.shape[:2]

    @staticmethod
    def load_images_and_anns(root_dir, label2idx):
        r"""
        Method to get the xml files and for each file
        get all the objects and their ground truth detection
        information for the dataset
        :param im_dir: Path of the images
        :param ann_dir: Path of annotation xmlfiles
        :param label2idx: Class Name to index mapping for dataset
        :return:
        """
        print('Classes:', label2idx)
        im_infos = []
        for ann_file in tqdm(glob.glob(os.path.join(root_dir, '*.xml'))):
            im_info = {}
            im_info['img_id'] = os.path.basename(ann_file).split('.xml')[0]
            im_info['filename'] = os.path.join(root_dir, '{}.jpg'.format(im_info['img_id']))
            ann_info = ET.parse(ann_file)
            root = ann_info.getroot()
            size = root.find('size')
            width = int(size.find('width').text)
            height = int(size.find('height').text)
            im_info['width'] = width
            im_info['height'] = height
            detections = []

            for obj in ann_info.findall('object'):
                det = {}
                label = label2idx[obj.find('name').text]
                bbox_info = obj.find('bndbox')
                bbox = [
                    (float(bbox_info.find('xmin').text) - 1) / width,
                    (float(bbox_info.find('ymin').text) - 1) / height,
                    (float(bbox_info.find('xmax').text) - 1) / width,
                    (float(bbox_info.find('ymax').text) - 1) / height
                ]
                det['label'] = label
                det['bbox'] = bbox
                det['bbox_coords'] = box2segment(bbox)
                detections.append(det)
            im_info['detections'] = detections
            im_infos.append(im_info)
        print('Total {} images found'.format(len(im_infos)))
        return im_infos

In [ ]:
class RefineDetVOCDataModule(L.LightningDataModule):
    def __init__(self, root_dir, classes, img_size=512, batch_size=4, num_workers=2, seed=42):
        super().__init__()
        self.root_dir = root_dir
        self.classes = classes
        self.img_size = img_size
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.seed = seed

    def prepare(self):
        # check dataset structure
        assert os.path.exists(self.root_dir) and os.path.isdir(self.root_dir), EOFError
        for mode in ['train', 'valid', 'test']:
            subfol = os.path.join(self.root_dir, mode)
            assert os.path.exists(subfol) and os.path.isdir(subfol), EOFError

    def setup(self, stage):
        self.stage = stage
        if self.stage == 'fit':
            self.train_dataset = RefineDetVOCDataset(mode='train', root_dir=self.root_dir, classes=self.classes, img_size=self.img_size)
            self.val_dataset = RefineDetVOCDataset(mode='valid', root_dir=self.root_dir, classes=self.classes, img_size=self.img_size)
        if self.stage == 'test':
            self.test_dataset = RefineDetVOCDataset(mode='test', root_dir=self.root_dir, classes=self.classes, img_size=self.img_size)

    def train_dataloader(self):
        return DataLoader(dataset=self.train_dataset,
                          batch_size=self.batch_size,
                          shuffle=True, drop_last=True,
                          num_workers=self.num_workers,
                          collate_fn=self.collate_fn)

    def val_dataloader(self):
        return DataLoader(dataset=self.val_dataset,
                          batch_size=self.batch_size,
                          shuffle=False, drop_last=False,
                          num_workers=self.num_workers,
                          collate_fn=self.collate_fn)

    def test_dataloader(self):
        return DataLoader(dataset=self.test_dataset,
                          batch_size=1, shuffle=False,
                          num_workers=self.num_workers,
                          collate_fn=self.collate_fn)

    def collate_fn(self, batch):
        if self.stage == 'fit':
            imgs, targets = tuple(zip(*batch))
            return torch.stack(imgs), targets
        else:
            return batch[0] # (img, target, img_file)

In [ ]:
# define and show train samples
train_dataset = RefineDetVOCDataset(
    mode='train', root_dir=ROOT_DIR, classes=CLASSES, img_size=IMG_SIZE
)

fig, ax = plt.subplots(nrows=4, ncols=4, figsize=(20, 20))
for i, sample_idx in enumerate(random.sample(range(len(train_dataset)), k=16)):
    im, targets = train_dataset[sample_idx]
    im = np.ascontiguousarray(np.uint8(im.permute(1, 2, 0).numpy()))
    labels = targets['labels'].numpy().astype('int64')
    bboxes = (train_dataset.img_size * targets['boxes']).numpy().astype('int64')

    for label, bbox in zip(labels, bboxes):
        h, w, _ = im.shape
        plot_one_box(img=im, bbox=bbox, label=f'{CLASSES[label]}: {label}')

    row, col = i // 4, i % 4
    ax[row][col].imshow(im)
plt.show()

In [ ]:
# define and show valid samples
val_dataset = RefineDetVOCDataset(
    mode='valid', root_dir=ROOT_DIR, classes=CLASSES, img_size=IMG_SIZE
)

fig, ax = plt.subplots(nrows=4, ncols=4, figsize=(20, 20))
for i, sample_idx in enumerate(random.sample(range(len(val_dataset)), k=16)):
    im, targets = val_dataset[sample_idx]
    im = np.ascontiguousarray(np.uint8(255 * im.permute(1, 2, 0).numpy()))
    labels = targets['labels'].numpy().astype('int64')
    bboxes = (val_dataset.img_size * targets['boxes']).numpy().astype('int64')

    for label, bbox in zip(labels, bboxes):
        h, w, _ = im.shape
        plot_one_box(img=im, bbox=bbox, label=f'{CLASSES[label]}: {label}')

    row, col = i // 4, i % 4
    ax[row][col].imshow(im)
plt.show()

In [ ]:
# show test samples
test_dataset = RefineDetVOCDataset(
    mode='test', root_dir=ROOT_DIR, classes=CLASSES, img_size=IMG_SIZE
)

fig, ax = plt.subplots(nrows=4, ncols=4, figsize=(20, 12))
for i, sample_idx in enumerate(random.sample(range(len(test_dataset)), k=16)):
    im, targets, img_file = test_dataset[sample_idx]
    labels, bboxes = targets['labels'], targets['boxes']

    for label, bbox in zip(labels, bboxes):
        h, w, _ = im.shape
        plot_one_box(img=im, bbox=bbox, label=f'{CLASSES[label]}: {label}')

    row, col = i // 4, i % 4
    ax[row][col].imshow(im)
plt.show()

In [ ]:
datamodule = RefineDetVOCDataModule(
    root_dir=ROOT_DIR,
    classes=CLASSES,
    img_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    num_workers=0
)

# III. Model Training

In [ ]:
from RefineDet.models.refinedet_cam import build_refinedet_cam
from RefineDet.models.refinedet_novel_cam import build_refinedet_novel_cam
from RefineDet.layers.modules import RefineDetMultiBoxLoss

In [ ]:
from torch import nn
from torchmetrics.detection import MeanAveragePrecision

In [ ]:
from lightning.pytorch.loggers import TensorBoardLogger
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping

In [ ]:
class RefineDetModel(L.LightningModule):
    def __init__(self, classes, img_size, novel=True, vgg_path=''):
        super().__init__()
        self.save_hyperparameters()
        # initialize model
        self.classes = classes
        self.img_size = img_size
        self.novel = novel
        if novel:
            self.detector = build_refinedet_novel_cam(size=img_size, num_classes=len(classes) + 1)
        else:
            self.detector = build_refinedet_cam(size=img_size, num_classes=len(classes) + 1)
        print('Initializing weights...')
        self.detector.extras.apply(self.weights_init)
        self.detector.arm_loc.apply(self.weights_init)
        self.detector.arm_conf.apply(self.weights_init)
        self.detector.odm_loc.apply(self.weights_init)
        self.detector.odm_conf.apply(self.weights_init)
        self.detector.tcb0.apply(self.weights_init)
        self.detector.tcb1.apply(self.weights_init)
        self.detector.tcb2.apply(self.weights_init)
        if os.path.exists(vgg_path):
            try:
                print('Pretrainted VGG loaded:', self.detector.vgg.load_state_dict(torch.load(vgg_path)))
            except Exception as e:
                print('Load Pretrained VGG fail: ', e)
        self.register_buffer('device_holder', torch.empty([]))

        # define criterion and metrics
        self.arm_criterion = RefineDetMultiBoxLoss(2, 0.5, True, 0, True, 3, 0.5, False)
        self.odm_criterion = RefineDetMultiBoxLoss(len(classes) + 1, 0.5, True, 0, True, 3, 0.5, False, use_ARM=True)
        self.metrics = MeanAveragePrecision()

    def forward(self, imgs, top_k=500, objectness_thre=0.01, conf_thresh=0.01, nms_thresh=0.45):
        assert not self.training, NotImplementedError('Please .eval() to run this method.')
        preds, _ = self.detector(imgs, return_det=True, top_k=top_k, objectness_thre=objectness_thre, conf_thresh=conf_thresh, nms_thresh=nms_thresh)
        results = []
        for i in range(len(imgs)):
            all_labels, all_boxes, all_scores = [], [], []
            for j in range(1, preds.size(1)):
                dets = preds[i, j, :]
                if dets.device != imgs.device:
                    dets = dets.to(imgs.device)
                mask = dets[:, 0].gt(0.).expand(5, dets.size(0)).t()
                dets = torch.masked_select(dets, mask).view(-1, 5)
                if dets.size(0) == 0:
                    continue
                all_labels.append(torch.full(size=(len(dets),), fill_value=j - 1, dtype=torch.long, device=dets.device))
                all_boxes.append(dets[:, 1:])
                all_scores.append(dets[:, 0])
            results.append({
                'labels': torch.concat(all_labels) if len(all_labels) else len(imgs) * [],
                'boxes': torch.concat(all_boxes) if len(all_boxes) else [],
                'scores': torch.concat(all_scores) if len(all_scores) else []
            })
        return results # Sequence[{labels, boxes, scores}]

    def predict(self, img, top_k=500, objectness_thre=0.01, conf_thresh=0.25, nms_thresh=0.45, visualize=False):
        im, *ratio_pad = letterbox(im=img, new_shape=2 * [self.img_size], auto=False)
        im = torch.from_numpy(im.astype('float32')).permute(2, 0, 1).unsqueeze(0).to(self.device_holder.device)
        with torch.no_grad():
            result = self(im, top_k=top_k, objectness_thre=objectness_thre, conf_thresh=conf_thresh, nms_thresh=nms_thresh)[0]
        result = {name: value.cpu().numpy() if len(value) else value for name, value in result.items()}
        if len(result['boxes']):
            result['boxes'] = scale_boxes(img1_shape=im.shape[2:], boxes=self.img_size * result['boxes'], img0_shape=img.shape[:2], ratio_pad=ratio_pad)
        if visualize:
            vis_img = img.copy()
            for l, bbox, conf in zip(result['labels'], result['boxes'], result['scores']):
                label = f'{self.classes[int(l)]}: {conf:.2f}'
                plot_one_box(vis_img, bbox, label=label)
            result['vis_img'] = vis_img
        return result # [label, boxes scores, vis_img]

    def training_step(self, batch, batch_idx):
        # load data
        images, targets = batch
        targets = [
            torch.concat([t['boxes'], t['labels'].type_as(t['boxes']).view(-1, 1)], dim=1) for t in targets
        ] # format = [xmin, ymin, xmax, ymax, label_idx] - note box's elements are normalized in [0, 1]
        # forward
        preds = self.detector(images)
        # compute loss
        arm_loss_l, arm_loss_c = self.arm_criterion(preds, targets)
        odm_loss_l, odm_loss_c = self.odm_criterion(preds, targets)
        arm_loss = arm_loss_l + arm_loss_c
        odm_loss = odm_loss_l + odm_loss_c
        loss = arm_loss + odm_loss
        self.log_dict({
            'arm_loc': arm_loss_l.item(),
            'arm_conf': arm_loss_c.item(),
            'odm_loc': odm_loss_l.item(),
            'odm_conf': odm_loss_c.item()
        }, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        # load data
        images, targets = batch
        targets = [
            torch.concat([t['boxes'], t['labels'].type_as(t['boxes']).view(-1, 1)], dim=1) for t in targets
        ] # format = [xmin, ymin, xmax, ymax, label_idx] - note box's elements are normalized in [0, 1]
        # forward
        preds = self.detector(images, return_det=False)
        # compute loss
        arm_loss_l, arm_loss_c = self.arm_criterion(preds, targets)
        odm_loss_l, odm_loss_c = self.odm_criterion(preds, targets)
        arm_loss = arm_loss_l + arm_loss_c
        odm_loss = odm_loss_l + odm_loss_c
        loss = arm_loss + odm_loss
        self.log('val_loss', loss.item(), prog_bar=True)
        return loss

    def fit(self, datamodule, epochs=200, accumulation=1, grad_clip_val=10.0, accelerator='gpu', devices=[0], patience=-1, project='runs', ckpt_path=None):
        grad_clip_algo = 'norm' if grad_clip_val is not None else None
        project = os.path.join(project, 'refine_det_novel' if self.novel else 'refine_det')
        callbacks = [
            ModelCheckpoint(save_top_k=0, save_last=True), # save last training states
            ModelCheckpoint(monitor='val_loss', mode='min', filename='best_{epoch}-{val_loss:.2f}', save_weights_only=True, save_top_k=3), # save weights of best models
        ]
        if patience != -1:
            callbacks.append(EarlyStopping(monitor='val_loss', min_delta=0.001, patience=patience)) # Early stopping the training process

        trainer = L.Trainer(
            logger=TensorBoardLogger(save_dir=project, name='train'),
            max_epochs=epochs, min_epochs=1,
            accumulate_grad_batches=accumulation,
            gradient_clip_val=grad_clip_val,
            gradient_clip_algorithm=grad_clip_algo,
            callbacks=callbacks,
            accelerator=accelerator,
            devices=devices,
            enable_progress_bar=True,
            enable_checkpointing=True,
            num_sanity_val_steps=3
        )
        trainer.fit(model=self, datamodule=datamodule, ckpt_path=ckpt_path)
        return self

    def configure_optimizers(self):
        # get needed training parameters
        epochs = self.trainer.max_epochs
        batch_size = self.trainer.datamodule.batch_size
        num_samples = len(self.trainer.datamodule.train_dataset)
        accumulation = self.trainer.accumulate_grad_batches
        # configure optimizer
        optimizer = torch.optim.SGD(
            params=filter(lambda p: p.requires_grad, self.detector.parameters()),
            lr=1e-3, weight_decay=5e-4, momentum=0.9
        )
        # configure learning rate scheduler
        lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer=optimizer, eta_min=1e-6, last_epoch=-1,
            T_max=epochs * (num_samples // batch_size)
        )
        return {
            'optimizer': optimizer,
            'lr_scheduler': {
                'scheduler': lr_scheduler,
                'interval': 'step',
                'frequency': accumulation
            }
        }

    def evaluate(self, datamodule, top_k=500, objectness_thre=0.01, conf_thresh=0.25, nms_thresh=0.45, iou_thresh=0.5, visualize=False, project='runs'):
        project = os.path.join(project, 'refine_det_novel' if self.novel else 'refine_det')
        exp_path = increment_path(Path(project) / 'eval' / 'version', mkdir=True)
        if visualize:
            (exp_path / 'plots').mkdir(exist_ok=True)
        datamodule.prepare()
        datamodule.setup(stage='test')

        class_names = self.classes
        preds = {cls_name: [] for cls_name in class_names}
        gts = {cls_name: {} for cls_name in class_names}
        for (img, targets, img_file) in tqdm(datamodule.test_dataloader(), desc='Evaluation'):
            # run predicting
            results = self.predict(img, top_k, objectness_thre, conf_thresh, nms_thresh, visualize)

            # store predictions
            scores, boxes, labels = results['scores'], results['boxes'], results['labels']
            for (conf, bbox, cls_idx) in zip(scores, boxes, labels):
                cls_name = class_names[int(cls_idx)]
                preds[cls_name].append({'stem': img_file, 'bbox': bbox, 'conf': conf})

            # store grounth truth
            for cls_name in class_names:
                gts[cls_name][img_file] = []
            for cls_idx, bbox in zip(targets['labels'], targets['boxes']):
                cls_name = class_names[int(cls_idx)]
                gts[cls_name][img_file].append(bbox)
            for cls_name in class_names:
                bbox = np.array(gts[cls_name][img_file])
                gts[cls_name][img_file] = {
                    'bbox': bbox, 'check': np.zeros(len(bbox), dtype=np.bool_)
                }

            # visualize image
            if visualize:
                vis_img = cv2.cvtColor(src=results['vis_img'], code=cv2.COLOR_RGB2BGR)
                cv2.imwrite((exp_path / 'plots' / img_file).as_posix(), vis_img)

        # run evaluation
        for cls_name in class_names:
            preds[cls_name] = sorted(preds[cls_name], key=lambda x: -x['conf'])
        eval_results = {
            'PR_eval': PR_eval(gts, preds, class_names, iou_thr=iou_thresh),
            'confusion_matrix': confusion_matrix(gts, preds, class_names, iou_thr=iou_thresh)
        }

        # cache results
        for cls_name in class_names:
            for i in range(len(preds[cls_name])):
                preds[cls_name][i]['bbox'] = preds[cls_name][i]['bbox'].tolist()
                preds[cls_name][i]['conf'] = float(preds[cls_name][i]['conf'])

            for img_file in gts[cls_name]:
                gts[cls_name][img_file]['bbox'] = gts[cls_name][img_file]['bbox'].tolist()
                gts[cls_name][img_file]['check'] = len(gts[cls_name][img_file]['bbox']) * [False]

        with open(exp_path / 'cache_results.json', mode='w') as file:
            json.dump(obj={'preds': preds, 'gts': gts}, fp=file, indent=4)

        cache_eval_results = deepcopy(eval_results)
        for cls_name in (class_names + ['Avg.']):
            for metric_name in ['rec_array', 'prec_array', 'rec', 'prec', 'ap', 'f1']:
                if cache_eval_results['PR_eval'][cls_name][metric_name].size > 1:
                    cache_eval_results['PR_eval'][cls_name][metric_name] = cache_eval_results['PR_eval'][cls_name][metric_name].tolist()
                else:
                    cache_eval_results['PR_eval'][cls_name][metric_name] = float(cache_eval_results['PR_eval'][cls_name][metric_name])
        for cls_name in (class_names + ['BACKGROUND']):
            cache_eval_results['confusion_matrix'][cls_name] = cache_eval_results['confusion_matrix'][cls_name].tolist()
        with open(exp_path / 'cache_eval_results.json', mode='w') as file:
            json.dump(obj=cache_eval_results, fp=file, indent=4)

        return eval_results

    @staticmethod
    def weights_init(m):
        if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
            nn.init.xavier_uniform_(m.weight.data)
            m.bias.data.zero_()

In [ ]:
# Train RefineDet model
refine_det = RefineDetModel(
    classes=CLASSES, img_size=IMG_SIZE, novel=False, vgg_path=VGG_PATH
)
refine_det.fit(datamodule=datamodule, epochs=EPOCHS, project=os.path.join(os.getcwd(), 'runs'), accelerator='gpu', devices=[0])

In [ ]:
# Train RefineDet Novel model
refine_det_novel = RefineDetModel(
    classes=CLASSES, img_size=IMG_SIZE, novel=True, vgg_path=VGG_PATH
)
refine_det_novel.fit(datamodule=datamodule, epochs=EPOCHS, project=os.path.join(os.getcwd(), 'runs'), accelerator='gpu', devices=[0])

# IV. Evaluation

## 4.1 Evaluate RefineDet

In [ ]:
# Evaluate RefineDet model
refine_det = RefineDetModel.load_from_checkpoint(
    checkpoint_path=os.path.join(os.getcwd(), 'runs/refine_det/train/version_0/checkpoints/best_epoch=46-val_loss=5.53.ckpt')
).eval().cuda()
eval_results = refine_det.evaluate(datamodule, conf_thresh=CONF_THRESH, nms_thresh=NMS_THRESH, iou_thresh=IOU_THRESH)

In [ ]:
pd.DataFrame(eval_results['PR_eval']).T[['rec', 'prec', 'ap', 'f1']]

In [ ]:
# Visualize Results
fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(16, 5))

# Draw PR curve metric
ax[0].plot(
    100 * eval_results['PR_eval']['Avg.']['rec_array'], 100 * eval_results['PR_eval']['Avg.']['prec_array'],
    label='RefineDet - mAP@{}: {:.3f}'.format(round(100 * IOU_THRESH), eval_results['PR_eval']['Avg.']['ap'])
)
ax[0].set_xlabel('Recall')
ax[0].set_ylabel('Precision')
ax[0].set_title('PR Curve')
ax[0].set_xlim(0, 100.1)
ax[0].set_ylim(0, 100.5)
ax[0].legend(loc='lower left')

# Draw confusion matrix
confusion_mat = eval_results['confusion_matrix']
sns.heatmap(np.array(list(confusion_mat.values())), cmap='Blues', annot=True, xticklabels=confusion_mat.keys(), yticklabels=confusion_mat.keys(), ax=ax[1])
ax[1].set_xlabel('Ground Truth')
ax[1].set_ylabel('Prediction')
ax[1].set_title('Confusion Matrix of RefineDet')

plt.show()

In [ ]:
img, targets, img_file = test_dataset[random.choice(range(len(test_dataset)))]

# visualize targets
vis_target_img = img.copy()
for l, bbox in zip(targets['labels'], targets['boxes']):
    plot_one_box(img=vis_target_img, bbox=bbox, label=CLASSES[int(l)])

# visualize predictions
vis_pred_img = refine_det.predict(img, conf_thresh=CONF_THRESH, nms_thresh=NMS_THRESH, visualize=True)['vis_img']

fig, ax = plt.subplots(ncols=3, figsize=(30, 5))
ax[0].imshow(img)
ax[1].imshow(vis_target_img)
ax[2].imshow(vis_pred_img)
plt.show()

## 4.2 Evaluate RefineDet Novel

In [ ]:
# Evaluate RefineDet Novel model
refine_det_novel = RefineDetModel.load_from_checkpoint(
    checkpoint_path=os.path.join(os.getcwd(), 'runs/refine_det_novel/train/version_0/checkpoints/best_epoch=59-val_loss=5.49.ckpt')
).eval().cuda()
eval_results = refine_det_novel.evaluate(datamodule, conf_thresh=CONF_THRESH, nms_thresh=NMS_THRESH, iou_thresh=IOU_THRESH)

In [ ]:
pd.DataFrame(eval_results['PR_eval']).T[['rec', 'prec', 'ap', 'f1']]

In [ ]:
# Visualize Results
fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(16, 5))

# Draw PR curve metric
ax[0].plot(
    100 * eval_results['PR_eval']['Avg.']['rec_array'], 100 * eval_results['PR_eval']['Avg.']['prec_array'],
    label='RefineDet Novel - mAP@{}: {:.3f}'.format(round(100 * IOU_THRESH), eval_results['PR_eval']['Avg.']['ap'])
)
ax[0].set_xlabel('Recall')
ax[0].set_ylabel('Precision')
ax[0].set_title('PR Curve')
ax[0].set_xlim(0, 100.1)
ax[0].set_ylim(0, 100.5)
ax[0].legend(loc='lower left')

# Draw confusion matrix
confusion_mat = eval_results['confusion_matrix']
sns.heatmap(np.array(list(confusion_mat.values())), cmap='Blues', annot=True, xticklabels=confusion_mat.keys(), yticklabels=confusion_mat.keys(), ax=ax[1])
ax[1].set_xlabel('Ground Truth')
ax[1].set_ylabel('Prediction')
ax[1].set_title('Confusion Matrix of RefineDet Novel')

plt.show()

In [ ]:
img, targets, img_file = test_dataset[random.choice(range(len(test_dataset)))]

# visualize targets
vis_target_img = img.copy()
for l, bbox in zip(targets['labels'], targets['boxes']):
    plot_one_box(img=vis_target_img, bbox=bbox, label=CLASSES[int(l)])

# visualize predictions
vis_pred_img = refine_det_novel.predict(img, conf_thresh=CONF_THRESH, nms_thresh=NMS_THRESH, visualize=True)['vis_img']

fig, ax = plt.subplots(ncols=3, figsize=(30, 5))
ax[0].imshow(img)
ax[1].imshow(vis_target_img)
ax[2].imshow(vis_pred_img)
plt.show()